# Lektion 16 - Implementering af skalerbare agenter med Microsoft Foundry

I denne notesbog bygger du en **produktionsklar kundesupportagent** til det fiktive firma **Contoso**. I modsætning til de tidligere lektioner er pointen her ikke agentens ræsonneringsloop — det er alt det, der er *rundt om* den, som gør en agent sikker at køre i skala:

1. **Værktøjskald** — ordreopslag og oprettelse af sager.
2. **RAG** — politik-svar fra en vidensdatabase.
3. **Hukommelse** — huske kunden på tværs af omgange.
4. **Modelrouting** — send simple forespørgsler til en lille model, komplekse til en stor model.
5. **Respons-caching** — besvar gentagne spørgsmål uden et modelkald.
6. **Menneskelig godkendelse** — refunderinger over en tærskel sætter pause for godkendelse.
7. **Evalueringsport** — et offline testsettet, der blokerer en dårlig udgivelse.
8. **Observerbarhed** — OpenTelemetry-tracing omkring hver forespørgsel.

Hvert afsnit er selvstændigt og kan køres. Læs hver linje — produktionsprimitive holdes bevidst små.


## Opsætning

Før du kører denne notebook, skal du sikre dig, at du har:

1. **Et Microsoft Foundry-projekt** med en udrullet chatmodel (f.eks. `gpt-4.1-mini`).
2. **Logget ind med Azure CLI** — kør `az login` i din terminal.
3. **Sat de nødvendige miljøvariabler:**
   - `AZURE_AI_PROJECT_ENDPOINT` — dit Microsoft Foundry-projekts endepunkt.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — navnet på din udrullede model.

RAG-sektionen bruger **Azure AI Search**, når `AZURE_SEARCH_SERVICE_ENDPOINT` og `AZURE_SEARCH_API_KEY` er sat, og falder tilbage til et in-memory søgning, så notebooken kan køre uden en Search-ressource.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import re
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential(),
)
print("Foundry client ready.")

## 1. Værktøjer

Produktionsværktøjer udfører reel arbejde mod reelle systemer. Her simulerer vi en ordredatabase og et ticketsystem med almindelige Python-funktioner. `@tool` dekoratøren eksponerer dem for agenten.

Bemærk, at `issue_refund` bruger `approval_mode="always_require"` for refunderinger over en tærskel — dette er den menneskelig-in-the-loop primitiv, vi bruger senere.


In [ ]:
# Simulated backend systems (in production these are API calls behind scoped identities).
ORDERS = {
    "A1001": {"status": "shipped", "total": 42.00, "eta": "2 days"},
    "A1002": {"status": "processing", "total": 128.50, "eta": "5 days"},
    "A1003": {"status": "delivered", "total": 19.99, "eta": "delivered"},
}
TICKETS: list[dict] = []
REFUND_APPROVAL_THRESHOLD = 50.0


@tool(approval_mode="never_require")
def get_order_status(order_id: Annotated[str, "The customer's order ID, e.g. A1001"]) -> str:
    """Look up the status of a customer order."""
    order = ORDERS.get(order_id.upper())
    if not order:
        return f"No order found with ID {order_id}."
    return (
        f"Order {order_id.upper()}: status={order['status']}, "
        f"total=${order['total']:.2f}, eta={order['eta']}."
    )


@tool(approval_mode="never_require")
def open_ticket(
    subject: Annotated[str, "Short subject line for the support ticket"],
    details: Annotated[str, "Full description of the customer's issue"],
) -> str:
    """Open a support ticket for issues that need human follow-up."""
    ticket_id = f"T{1000 + len(TICKETS) + 1}"
    TICKETS.append({"id": ticket_id, "subject": subject, "details": details})
    return f"Ticket {ticket_id} opened: {subject}"


def refund_needs_approval(amount: float) -> bool:
    """Refunds above the threshold require a human approver."""
    return amount > REFUND_APPROVAL_THRESHOLD


@tool(approval_mode="always_require")
def issue_refund(
    order_id: Annotated[str, "The order to refund"],
    amount: Annotated[float, "Refund amount in USD"],
) -> str:
    """Issue a refund. Execution pauses for human approval before it runs."""
    return f"Refund of ${amount:.2f} issued for order {order_id.upper()}."


print("Tools defined.")

## 2. RAG — Politiks vidensbase

Politikspørgsmål ("hvad er din returpolitik?") bør besvares fra en autoritativ kilde, ikke modellens hukommelse. Vi pakker en lille vidensbase ind som et søgeværktøj.

I produktion er dette **Azure AI Search**; her giver vi en nøgleordssøgning i hukommelsen, så notebooken kan køre overalt, og skifter automatisk til Azure AI Search, når miljøvariablerne er til stede.


In [ ]:
KNOWLEDGE_BASE = {
    "returns": "Contoso accepts returns within 30 days of delivery for a full refund. Items must be unused and in original packaging.",
    "shipping": "Standard shipping takes 3-5 business days. Express shipping (1-2 days) is available at checkout for an extra fee.",
    "warranty": "All Contoso electronics carry a 12-month limited warranty covering manufacturing defects.",
    "refund_policy": "Refunds are processed to the original payment method within 5 business days of approval. Refunds over $50 require a supervisor's approval.",
}


def _in_memory_search(query: str) -> str:
    q = query.lower()
    hits = [text for key, text in KNOWLEDGE_BASE.items() if key.replace("_", " ") in q or key in q]
    if not hits:
        # crude keyword fallback so the tool still returns something useful
        hits = [text for text in KNOWLEDGE_BASE.values() if any(w in text.lower() for w in q.split())]
    return "\n".join(hits) if hits else "No matching policy found."


def _azure_search(query: str) -> str:
    from azure.core.credentials import AzureKeyCredential
    from azure.search.documents import SearchClient

    client = SearchClient(
        endpoint=os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"],
        index_name=os.getenv("AZURE_SEARCH_INDEX_NAME", "contoso-policies"),
        credential=AzureKeyCredential(os.environ["AZURE_SEARCH_API_KEY"]),
    )
    results = client.search(search_text=query, top=3)
    return "\n".join(r.get("content", "") for r in results) or "No matching policy found."


USE_AZURE_SEARCH = bool(os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT") and os.getenv("AZURE_SEARCH_API_KEY"))


@tool(approval_mode="never_require")
def search_policies(query: Annotated[str, "The policy question to look up"]) -> str:
    """Search Contoso support policies to answer customer questions."""
    if USE_AZURE_SEARCH:
        return _azure_search(query)
    return _in_memory_search(query)


print(f"RAG ready. Using {'Azure AI Search' if USE_AZURE_SEARCH else 'in-memory search'}.")

## 3. Hukommelse

En supportagent, der glemmer, hvem den taler med, er en dårlig supportagent. Vi opbevarer en lille profildatabase pr. kunde og indsætter et kort resumé i agentens instruktioner. I produktion er dette en hukommelsestjeneste (se Lektion 13); her gør en ordbog mønsteret synligt.


In [ ]:
CUSTOMER_MEMORY: dict[str, dict] = {
    "cust-42": {"name": "Dana", "tier": "enterprise", "recent_order": "A1002"},
    "cust-99": {"name": "Sam", "tier": "standard", "recent_order": "A1003"},
}


def memory_context(customer_id: str) -> str:
    profile = CUSTOMER_MEMORY.get(customer_id)
    if not profile:
        return "This is a new customer with no history."
    return (
        f"Customer {profile['name']} ({profile['tier']} tier). "
        f"Most recent order: {profile['recent_order']}."
    )


print(memory_context("cust-42"))

## 4 & 5. Model Routing og Respons Caching

To omkostningshåndtag forbundet til en enkelt forespørgselsbehandler:

- **Routing**: en enkel heuristisk klassifikator afgør, om en forespørgsel har brug for den lille eller den store model.
- **Caching**: normaliserede gentagne spørgsmål serveres direkte fra en cache uden modelkald.

Klassifikatoren her er bevidst simpel. I produktion ville du validere den mod trafikken og kunne erstatte den med Foundrys Model Router.


In [ ]:
SMALL_MODEL = os.getenv("AZURE_AI_SMALL_MODEL", model)   # e.g. gpt-4.1-mini
LARGE_MODEL = os.getenv("AZURE_AI_LARGE_MODEL", model)   # e.g. gpt-4.1

response_cache: dict[str, str] = {}
route_counters = {"small": 0, "large": 0, "cache": 0}


def normalize(query: str) -> str:
    return re.sub(r"\s+", " ", query.lower().strip())


COMPLEX_SIGNALS = ("refund", "cancel", "complaint", "escalate", "broken", "wrong", "why")


def is_simple(query: str) -> bool:
    """Route complex or high-stakes requests to the large model; everything else to the small one."""
    q = query.lower()
    if any(signal in q for signal in COMPLEX_SIGNALS):
        return False
    return len(q.split()) <= 20


def choose_model(query: str) -> str:
    return SMALL_MODEL if is_simple(query) else LARGE_MODEL


print(f"Small model: {SMALL_MODEL} | Large model: {LARGE_MODEL}")

## 6 & 8. Agenten, menneskelig godkendelse og observerbarhed

Nu samler vi agenten ud fra værktøjerne ovenfor og pakker hver forespørgsel ind i en OpenTelemetry-span. Funktionen `handle_support_request` er produktionsforespørgselsbehandleren: cache → rute → trace → kør → cache.

Menneskelig godkendelse håndteres af frameworket: fordi `issue_refund` har `approval_mode="always_require"`, sættes kørslen på pause og viser en godkendelsesanmodning, som vi eksplicit løser.


In [ ]:
# Tracing: use the Agent Framework tracer if available, else a no-op so the notebook runs anywhere.
try:
    from agent_framework.observability import get_tracer
    tracer = get_tracer()
except Exception:  # observability extras not installed
    from contextlib import contextmanager

    class _NoopSpan:
        def set_attribute(self, *_args, **_kwargs):
            pass

    class _NoopTracer:
        @contextmanager
        def start_as_current_span(self, _name):
            yield _NoopSpan()

    tracer = _NoopTracer()


SUPPORT_INSTRUCTIONS = (
    "You are Contoso's customer support agent. Be concise, friendly, and accurate. "
    "Use search_policies for policy questions, get_order_status for orders, "
    "open_ticket when a human needs to follow up, and issue_refund for refunds. "
    "Never invent policy details."
)

support_agent = provider.as_agent(
    name="ContosoSupportAgent",
    instructions=SUPPORT_INSTRUCTIONS,
    tools=[get_order_status, open_ticket, search_policies, issue_refund],
)


async def handle_support_request(query: str, customer_id: str) -> str:
    # 1. Serve from cache when we can.
    key = normalize(query)
    if key in response_cache:
        route_counters["cache"] += 1
        return response_cache[key]

    # 2. Route by complexity to control cost.
    chosen_model = choose_model(query)
    route_counters["small" if chosen_model == SMALL_MODEL else "large"] += 1

    # 3. Add per-customer memory to the prompt.
    context = memory_context(customer_id)
    prompt = f"[Customer context: {context}]\n\n{query}"

    # 4. Run inside a trace span for observability.
    with tracer.start_as_current_span("support_request") as span:
        span.set_attribute("customer.id", customer_id)
        span.set_attribute("routed.model", chosen_model)
        response = await support_agent.run(prompt, model=chosen_model)

    text = response.text
    response_cache[key] = text
    return text


print("Support agent assembled.")

In [ ]:
# Try a few requests. The first is simple (small model), the second is a refund (large model + approval path).
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print(await handle_support_request("Where is my order A1002?", "cust-42"))
print("---")
# Repeat the first question -> served from cache.
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print("Routing counters:", route_counters)

## 7. Evalueringsport

Dette er udgivelsesporten fra lektionen: et offline test sæt scorer agenten, og implementering fortsætter kun, hvis beståelsesprocenten overstiger en tærskel. Scoreren her er en simpel nøgleordsoverlapningskontrol for at holde notebogen selvstændig; i produktion ville du bruge en LLM-som-dommer eller en rammevurderingsværktøj (se Lektion 10).


In [ ]:
TEST_CASES = [
    {"input": "How long do I have to return an item?", "expected": ["30 days", "refund"]},
    {"input": "How fast is standard shipping?", "expected": ["3-5", "business days"]},
    {"input": "What is the status of order A1001?", "expected": ["shipped", "A1001"]},
    {"input": "Do your electronics have a warranty?", "expected": ["12-month", "warranty"]},
]


def score_response(actual: str, expected_keywords: list[str]) -> float:
    actual_l = actual.lower()
    hits = sum(1 for kw in expected_keywords if kw.lower() in actual_l)
    return hits / len(expected_keywords)


async def evaluation_gate(test_cases: list[dict], threshold: float = 0.8) -> bool:
    passed = 0
    for case in test_cases:
        result = await support_agent.run(case["input"])
        s = score_response(result.text, case["expected"])
        status = "PASS" if s >= 0.5 else "FAIL"
        print(f"[{status}] {case['input']}  (score={s:.0%})")
        if s >= 0.5:
            passed += 1
    pass_rate = passed / len(test_cases)
    print(f"\nEvaluation pass rate: {pass_rate:.0%} (gate: {threshold:.0%})")
    return pass_rate >= threshold


gate_passed = await evaluation_gate(TEST_CASES, threshold=0.8)
print("\nDeploy allowed:" , gate_passed)

## At samle det hele: En simuleret udgivelse

Cellen nedenfor viser hele løkken, som lektionen beskriver: kør evalueringsporten, og "udrul" kun, hvis den bestås. Dette er det mønster, du ville køre i CI, før du promoverer en agentversion til Foundry Agent Service.


In [ ]:
async def release(test_cases: list[dict]) -> None:
    print("Running pre-deployment evaluation gate...\n")
    if await evaluation_gate(test_cases, threshold=0.8):
        print("\n✅ Gate passed — promoting agent version to the Foundry Agent Service.")
    else:
        print("\n❌ Gate failed — release blocked. Fix the agent and re-run.")


await release(TEST_CASES)

## Resumé

Du har samlet en produktionsklar kundesupportagent med alle operationelle aspekter indbygget:

- **Værktøjer, RAG og hukommelse** giver agenten kapacitet og kontekst.
- **Modelruting og caching** holder latenstid og omkostninger under kontrol.
- **Menneskelig godkendelse** vogter over højrisikohandlinger som store refusioner.
- **Evalueringporten** blokerer dårlige udgivelser inden de sendes ud.
- **Sporing** gør hver forespørgsel observerbar.

### Udfordring

Udvid denne agent til:

1. **Understøtte flere modeller** — tilføj et tredje "reasoning" lag og rut eskaleringer/klager til det.
2. **Tilføje evalueringsporte** — udvid `TEST_CASES` til at inkludere scenarier for refusionsgodkendelse og bekræft porten fanger regressioner.
3. **Tilføje omkostningsbevidst routing** — spor en estimeret omkostning pr. forespørgsel (lille vs stor vs cache) og udskriv en omkostningsrapport efter en batch af blandede forespørgsler.

I den næste lektion tager du den modsatte rejse og kører en agent helt på din egen maskine med Microsoft Foundry Local og Qwen.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokument er blevet oversat ved hjælp af AI-oversættelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selvom vi bestræber os på nøjagtighed, skal du være opmærksom på, at automatiserede oversættelser kan indeholde fejl eller unøjagtigheder. Det originale dokument på dets oprindelige sprog bør betragtes som den autoritative kilde. For kritisk information anbefales professionel menneskelig oversættelse. Vi påtager os intet ansvar for misforståelser eller fejltolkninger, der opstår som følge af brugen af denne oversættelse.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
